# Gromov-Wasserstein Causal Abstractions (Base-10 One-Hot)

This notebook aligns SCM variables to neural sites using intervention-effect geometries and
entropic Gromov-Wasserstein coupling.

Updates in this version:
- Two-digit base-10 addition with one-hot digit inputs (`40` dims total)
- Output classes `0..198` (`200` classes)
- PCA-rotation interventions with prefix subspaces `[0..i-1]`
- Site catalog = every layer × every prefix size `i=1..k`


In [ ]:
import os
import random
import itertools
import inspect
from dataclasses import dataclass
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score

from pyvene import CausalModel
from pyvene import IntervenableModel
from pyvene import RepresentationConfig, IntervenableConfig
from pyvene import PCARotatedSpaceIntervention
from pyvene.models.mlp.modelings_mlp import MLPConfig as PyveneMLPConfig
from variable_width_mlp import (
    VariableWidthMLPConfig,
    VariableWidthMLPForClassification,
    logits_from_output,
    load_variable_width_mlp_checkpoint,
)

try:
    import ot  # POT: Python Optimal Transport
except Exception as exc:
    raise ImportError("This notebook requires POT (`pip install pot`).") from exc

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)
print("POT version:", ot.__version__)


## Experiment Configuration


In [ ]:
# Optional factual sanity test size for the loaded model.
N_FACTUAL_TEST = 4000
RUN_LOAD_SANITY_EVAL = True

# Pair-bank sizes for alignment and held-out validation.
N_ALIGNMENT_PAIRS = 4096
N_HELDOUT_PAIRS = 1024
COLLECT_BATCH_SIZE = 256

# Checkpoint produced by addition_train.ipynb.
MODEL_CHECKPOINT_PATH = "artifacts/addition_mlp.pt"

# Variables to align.
TARGET_VARS = ["S1", "C1", "S2", "C2"]

# Site search: every selected layer, and every prefix i in 1..PCA_PREFIX_K.
PCA_PREFIX_K = 32
SEARCH_LAYER_INDICES = None  # None => all layers
SITE_UNIT = "pos"
SITE_POSITION = 0
SITE_MAX_UNITS = 1

# Cost-matrix construction.
GEOMETRY_METRIC = "cosine"   # "euclidean" or "cosine"
NORMALIZE_COST_MATRICES = True   # strongly recommended for euclidean

# GW defaults.
GW_CFG = {
    "loss_fun": "square_loss",
    "epsilon": 5e-2,  # larger default helps avoid underflow with high-dim euclidean costs
    "max_iter": 500,
    "tol": 1e-9,
    "verbose": False,
    "epsilon_retry_multipliers": [1, 5, 10, 50, 100],
}

TOP_K_VALIDATE = 3

# Verify pyvene's built-in MLPConfig is fixed-width (single h_dim + n_layer).
print("pyvene MLPConfig signature:", inspect.signature(PyveneMLPConfig.__init__))


## SCM: Base-10 Addition with Carry


In [ ]:
one_hot_digits = [np.eye(10, dtype=np.float32)[d] for d in range(10)]


def as_digit(x):
    arr = np.array(x).reshape(-1)
    if arr.size == 1:
        return int(arr[0])
    return int(arr.argmax())


variables = ["A1", "B1", "A2", "B2", "S1", "C1", "T2", "S2", "C2", "O"]

values = {
    "A1": one_hot_digits,
    "B1": one_hot_digits,
    "A2": one_hot_digits,
    "B2": one_hot_digits,
    "S1": list(range(10)),
    "C1": [0, 1],
    "T2": list(range(19)),
    "S2": list(range(10)),
    "C2": [0, 1],
    "O": list(range(200)),
}

parents = {
    "A1": [],
    "B1": [],
    "A2": [],
    "B2": [],
    "S1": ["A1", "B1"],
    "C1": ["A1", "B1"],
    "T2": ["A2", "B2"],
    "S2": ["T2", "C1"],
    "C2": ["T2", "C1"],
    "O": ["C2", "S2", "S1"],
}


def FILLER():
    return one_hot_digits[0]


functions = {
    "A1": FILLER,
    "B1": FILLER,
    "A2": FILLER,
    "B2": FILLER,
    "S1": lambda a1, b1: (as_digit(a1) + as_digit(b1)) % 10,
    "C1": lambda a1, b1: (as_digit(a1) + as_digit(b1)) // 10,
    "T2": lambda a2, b2: as_digit(a2) + as_digit(b2),
    "S2": lambda t2, c1: (int(t2) + int(c1)) % 10,
    "C2": lambda t2, c1: (int(t2) + int(c1)) // 10,
    "O": lambda c2, s2, s1: int(100 * int(c2) + 10 * int(s2) + int(s1)),
}

addition_model = CausalModel(variables, values, parents, functions)


In [ ]:
# SCM sanity checks over all 10,000 combinations.

def assignment_from_digits(digits):
    a1, b1, a2, b2 = [int(x) for x in digits]
    return {
        "A1": one_hot_digits[a1],
        "B1": one_hot_digits[b1],
        "A2": one_hot_digits[a2],
        "B2": one_hot_digits[b2],
    }

for digits in itertools.product(range(10), repeat=4):
    setting = addition_model.run_forward(assignment_from_digits(digits))
    a1, b1, a2, b2 = digits

    s1 = (a1 + b1) % 10
    c1 = (a1 + b1) // 10
    s2 = (a2 + b2 + c1) % 10
    c2 = (a2 + b2 + c1) // 10
    o = 100 * c2 + 10 * s2 + s1

    assert int(setting["S1"]) == s1
    assert int(setting["C1"]) == c1
    assert int(setting["S2"]) == s2
    assert int(setting["C2"]) == c2
    assert int(setting["O"]) == o

print("SCM truth-table checks passed.")


## Load Pretrained Variable-Width MLP


In [ ]:
if not os.path.exists(MODEL_CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"Checkpoint not found at {MODEL_CHECKPOINT_PATH}. Run addition_train.ipynb first."
    )

trained, loaded_cfg, checkpoint = load_variable_width_mlp_checkpoint(
    MODEL_CHECKPOINT_PATH,
    device,
)

print("Loaded checkpoint:", MODEL_CHECKPOINT_PATH)
print("Hidden dims:", loaded_cfg.hidden_dims)
print("n_layer:", loaded_cfg.n_layer)

# Optional factual sanity check.
# Use pyvene-generated input_ids directly so this check is agnostic to input permutation.
if RUN_LOAD_SANITY_EVAL:
    rng = np.random.default_rng(SEED + 55)

    def sanity_sampler():
        digits = rng.integers(0, 10, size=(4,), dtype=np.int64)
        return assignment_from_digits(digits)

    test_examples = addition_model.generate_factual_dataset(N_FACTUAL_TEST, sanity_sampler)
    X_test = torch.stack([ex["input_ids"] for ex in test_examples]).to(torch.float32)
    y_test = torch.tensor([int(ex["labels"].item()) for ex in test_examples], dtype=torch.long)

    preds = []
    with torch.no_grad():
        for start in range(0, N_FACTUAL_TEST, COLLECT_BATCH_SIZE):
            end = min(start + COLLECT_BATCH_SIZE, N_FACTUAL_TEST)
            logits = trained(inputs_embeds=X_test[start:end].to(device).unsqueeze(1))[0]
            preds.append(logits.detach().cpu().argmax(dim=1))
    preds = torch.cat(preds)

    print(f"Loaded-model factual accuracy: {accuracy_score(y_test.numpy(), preds.numpy()):.4f}")


## Pair Bank Interfaces


In [ ]:
@dataclass
class SiteSpec:
    layer: int
    prefix_dim: int
    hidden_dim: int
    component: str
    unit: str
    position: int
    max_units: int
    pca: object
    pca_mean: np.ndarray
    pca_std: np.ndarray


CANONICAL_VAR_ORDER = ["A1", "B1", "A2", "B2"]


def infer_input_var_order(causal_model, one_hot_digits):
    """
    Infer how pyvene packs leaf variables into input_ids.

    We assign unique marker digits to each leaf variable, generate one factual item,
    then read each 10-d segment argmax to recover variable order.
    """
    marker_digits = {
        "A1": 1,
        "B1": 2,
        "A2": 3,
        "B2": 4,
    }
    marker_to_var = {v: k for k, v in marker_digits.items()}

    marker_assignment = {
        var: one_hot_digits[d] for var, d in marker_digits.items()
    }

    example = causal_model.generate_factual_dataset(1, lambda: marker_assignment)[0]
    vec = example["input_ids"].detach().cpu().numpy().reshape(-1)

    if vec.size % 10 != 0:
        raise ValueError(f"Unexpected input_ids size {vec.size}; expected multiple of 10")

    inferred = []
    for seg_i in range(vec.size // 10):
        seg = vec[seg_i * 10 : (seg_i + 1) * 10]
        marker = int(seg.argmax())
        if marker not in marker_to_var:
            raise ValueError(
                f"Could not map segment {seg_i} marker {marker} to a leaf variable"
            )
        inferred.append(marker_to_var[marker])

    if sorted(inferred) != sorted(CANONICAL_VAR_ORDER):
        raise ValueError(
            f"Inferred order {inferred} does not match expected leaves {CANONICAL_VAR_ORDER}"
        )

    return inferred


INPUT_VAR_ORDER = infer_input_var_order(addition_model, one_hot_digits)
print("pyvene input_ids variable order:", INPUT_VAR_ORDER)


def digits_to_assignment(vec):
    # vec is interpreted in canonical order [A1, B1, A2, B2].
    a1, b1, a2, b2 = [int(x) for x in np.array(vec).reshape(-1)]
    return {
        "A1": one_hot_digits[a1],
        "B1": one_hot_digits[b1],
        "A2": one_hot_digits[a2],
        "B2": one_hot_digits[b2],
    }


def digits_to_inputs_embeds(digits_np, input_var_order=INPUT_VAR_ORDER):
    """
    Convert canonical digit rows [A1,B1,A2,B2] to model input vectors using
    the exact variable packing order pyvene uses for input_ids.
    """
    if digits_np.shape[1] != len(CANONICAL_VAR_ORDER):
        raise ValueError(f"Expected digits shape [N,4], got {digits_np.shape}")

    col_by_var = {var: i for i, var in enumerate(CANONICAL_VAR_ORDER)}
    ordered_digits = np.stack(
        [digits_np[:, col_by_var[var]] for var in input_var_order],
        axis=1,
    )

    onehots = np.eye(10, dtype=np.float32)[ordered_digits]  # [N, 4, 10]
    flat = onehots.reshape(digits_np.shape[0], 40)
    return torch.tensor(flat, dtype=torch.float32)


def build_pair_bank(n_pairs, seed):
    """
    Builds n_pairs of (base, source) digit tuples, and computes their SCM labels for the base and counterfactual settings.
    The counterfactual outputs are computed by intervening on each TARGET_VAR in the base assignment 
    with the corresponding value from the source assignment.
    """
    rng = np.random.default_rng(seed)
    base_digits = rng.integers(0, 10, size=(n_pairs, 4), dtype=np.int64)
    source_digits = rng.integers(0, 10, size=(n_pairs, 4), dtype=np.int64)

    scm_base = np.zeros(n_pairs, dtype=np.int64)
    scm_cf = {v: np.zeros(n_pairs, dtype=np.int64) for v in TARGET_VARS}

    for i in range(n_pairs):
        base_assignment = digits_to_assignment(base_digits[i])
        source_assignment = digits_to_assignment(source_digits[i])

        factual_setting = addition_model.run_forward(base_assignment)
        scm_base[i] = int(factual_setting["O"])

        for v in TARGET_VARS:
            cf_setting = addition_model.run_interchange(base_assignment, {v: source_assignment})
            scm_cf[v][i] = int(cf_setting["O"])

    return {
        "base_inputs": digits_to_inputs_embeds(base_digits),
        "source_inputs": digits_to_inputs_embeds(source_digits),
        "scm_base_label": torch.tensor(scm_base, dtype=torch.long),
        "scm_cf_label_by_var": {
            v: torch.tensor(vals, dtype=torch.long) for v, vals in scm_cf.items()
        },
    }


In [ ]:
alignment_bank = build_pair_bank(N_ALIGNMENT_PAIRS, seed=SEED + 1)
heldout_bank = build_pair_bank(N_HELDOUT_PAIRS, seed=SEED + 2)

print("Alignment bank base shape:", tuple(alignment_bank["base_inputs"].shape))
print("Alignment bank source shape:", tuple(alignment_bank["source_inputs"].shape))
print("Heldout bank base shape:", tuple(heldout_bank["base_inputs"].shape))

assert alignment_bank["base_inputs"].shape == (N_ALIGNMENT_PAIRS, 40)
assert alignment_bank["source_inputs"].shape == (N_ALIGNMENT_PAIRS, 40)
for v in TARGET_VARS:
    assert alignment_bank["scm_cf_label_by_var"][v].shape == (N_ALIGNMENT_PAIRS,)

# Determinism check.
bank_a = build_pair_bank(16, seed=SEED + 99)
bank_b = build_pair_bank(16, seed=SEED + 99)
assert torch.equal(bank_a["base_inputs"], bank_b["base_inputs"])
assert torch.equal(bank_a["source_inputs"], bank_b["source_inputs"])
assert torch.equal(bank_a["scm_base_label"], bank_b["scm_base_label"])
print("Pair-bank integrity checks passed.")


## PCA-Prefix Site Catalog and Intervention Collection


In [ ]:
def collect_layer_activations(model, inputs_flat, layer_idx, batch_size, device):
    # Returns [N, hidden_dim_of_layer] activations at token position 0.
    acts = []
    n = inputs_flat.shape[0]
    with torch.no_grad():
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            hidden = inputs_flat[start:end].to(device).unsqueeze(1)
            for i, block in enumerate(model.h):
                hidden = block(hidden)
                if i == layer_idx:
                    acts.append(hidden[:, 0, :].detach().cpu())
                    break
    return torch.cat(acts, dim=0).numpy()


def fit_layer_pca(acts):
    mean = acts.mean(axis=0)
    std = acts.std(axis=0) + 1e-6
    pca = PCA(n_components=acts.shape[1], svd_solver="full")
    pca.fit((acts - mean) / std)
    return pca, mean, std


def build_site_specs(model, base_inputs, batch_size, device, prefix_k, search_layers=None):
    hidden_dims = list(model.config.hidden_dims)
    layer_ids = list(range(len(hidden_dims))) if search_layers is None else [int(x) for x in search_layers]

    specs = []
    for layer_i in layer_ids:
        width = int(hidden_dims[layer_i])
        acts = collect_layer_activations(model, base_inputs, layer_i, batch_size, device)
        pca, mean, std = fit_layer_pca(acts)

        max_prefix = min(int(prefix_k), width)
        for i in range(1, max_prefix + 1):
            specs.append(
                SiteSpec(
                    layer=layer_i,
                    prefix_dim=i,
                    hidden_dim=width,
                    component=f"h[{layer_i}].output",
                    unit=SITE_UNIT,
                    position=SITE_POSITION,
                    max_units=SITE_MAX_UNITS,
                    pca=pca,
                    pca_mean=mean,
                    pca_std=std,
                )
            )
    return specs


def collect_model_outputs(trained, pair_bank, site_specs, batch_size, device):
    base_inputs = pair_bank["base_inputs"]
    source_inputs = pair_bank["source_inputs"]
    n = base_inputs.shape[0]

    base_logits_chunks = []
    with torch.no_grad():
        for start in tqdm(range(0, n, batch_size), desc="Base forward"):
            end = min(start + batch_size, n)
            out = trained(inputs_embeds=base_inputs[start:end].to(device).unsqueeze(1))
            base_logits_chunks.append(logits_from_output(out).detach().cpu())
    base_logits = torch.cat(base_logits_chunks, dim=0)

    num_sites = len(site_specs)
    num_classes = base_logits.shape[-1]
    site_cf_logits = torch.zeros((num_sites, n, num_classes), dtype=torch.float32)

    for site_idx, site in enumerate(tqdm(site_specs, desc="Site interventions")):
        pca_intervention = PCARotatedSpaceIntervention(
            embed_dim=site.hidden_dim,
            pca=site.pca,
            pca_mean=site.pca_mean,
            pca_std=site.pca_std,
        )

        config_site = IntervenableConfig(
            model_type=type(trained),
            representations=[
                RepresentationConfig(
                    layer=site.layer,
                    component=site.component,
                    unit=site.unit,
                    max_number_of_units=site.max_units,
                    intervention=pca_intervention,
                )
            ],
        )

        intervenable = IntervenableModel(config_site, trained, use_fast=True)
        intervenable.set_device(device)
        intervenable.disable_model_gradients()
        intervenable.disable_intervention_gradients()

        cf_chunks = []
        with torch.no_grad():
            for start in range(0, n, batch_size):
                end = min(start + batch_size, n)
                bsz = end - start

                base_batch = base_inputs[start:end].to(device).unsqueeze(1)
                source_batch = source_inputs[start:end].to(device).unsqueeze(1)

                pos_list = [[site.position]] * bsz
                prefix_subspace = [list(range(site.prefix_dim))] * bsz

                _, cf_out = intervenable(
                    {"inputs_embeds": base_batch},
                    [{"inputs_embeds": source_batch}],
                    {"sources->base": ([pos_list], [pos_list])},
                    subspaces=[prefix_subspace],
                )
                cf_chunks.append(logits_from_output(cf_out).detach().cpu())

        site_cf_logits[site_idx] = torch.cat(cf_chunks, dim=0)

    return {
        "base_logits": base_logits,
        "site_cf_logits": site_cf_logits,
    }


In [ ]:
site_specs = build_site_specs(
    trained,
    alignment_bank["base_inputs"],
    batch_size=COLLECT_BATCH_SIZE,
    device=device,
    prefix_k=PCA_PREFIX_K,
    search_layers=SEARCH_LAYER_INDICES,
)

print("Candidate site count:", len(site_specs))
print("First few sites:", site_specs[: min(8, len(site_specs))])

alignment_outputs = collect_model_outputs(
    trained=trained,
    pair_bank=alignment_bank,
    site_specs=site_specs,
    batch_size=COLLECT_BATCH_SIZE,
    device=device,
)

print("base_logits shape:", tuple(alignment_outputs["base_logits"].shape))
print("site_cf_logits shape:", tuple(alignment_outputs["site_cf_logits"].shape))

assert alignment_outputs["base_logits"].shape[0] == N_ALIGNMENT_PAIRS
assert alignment_outputs["site_cf_logits"].shape[:2] == (len(site_specs), N_ALIGNMENT_PAIRS)
assert not torch.isnan(alignment_outputs["base_logits"]).any()
assert not torch.isnan(alignment_outputs["site_cf_logits"]).any()
print("Intervention pipeline checks passed.")


## Effect Signatures and Geometry Matrices


In [ ]:
def build_effect_signatures(pair_bank, model_outputs, target_vars):
    """
    Build intervention-effect signatures for both abstract variables and neural sites.

    - Variable effects come from SCM interchanges: onehot(cf_label) - onehot(base_label)
    - Site effects come from neural interchanges: softmax(site_cf_logits) - softmax(base_logits)

    Signatures are kept at full resolution [C, N] and then flattened to [C*N],
    i.e. concatenation over the full dataset (no mean aggregation).
    """
    base_logits = model_outputs["base_logits"]
    site_cf_logits = model_outputs["site_cf_logits"]
    num_classes = int(base_logits.shape[-1])

    # SCM variable effects on the same (base, source) pairs.
    base_label_onehot = F.one_hot(pair_bank["scm_base_label"], num_classes=num_classes).to(torch.float32)
    var_effects = []
    for var in target_vars:
        cf_onehot = F.one_hot(pair_bank["scm_cf_label_by_var"][var], num_classes=num_classes).to(torch.float32)
        var_effects.append(cf_onehot - base_label_onehot)
    var_effects = torch.stack(var_effects, dim=0)  # [V, N, C]

    # Neural site effects on the same (base, source) pairs.
    base_prob = torch.softmax(base_logits, dim=-1)
    site_prob = torch.softmax(site_cf_logits, dim=-1)
    site_effects = site_prob - base_prob.unsqueeze(0)  # [S, N, C]

    # Re-orient to [*, C, N] so each signature is explicitly a (C x N) matrix.
    var_effect_mats = var_effects.permute(0, 2, 1).contiguous()   # [V, C, N]
    site_effect_mats = site_effects.permute(0, 2, 1).contiguous() # [S, C, N]

    # Concatenate by flattening [C, N] -> [C*N] for pairwise distance computation.
    var_signatures = var_effect_mats.reshape(var_effect_mats.shape[0], -1)    # [V, C*N]
    site_signatures = site_effect_mats.reshape(site_effect_mats.shape[0], -1) # [S, C*N]

    return {
        "var_effects": var_effects,
        "site_effects": site_effects,
        "var_effect_mats": var_effect_mats,
        "site_effect_mats": site_effect_mats,
        "var_signatures": var_signatures,
        "site_signatures": site_signatures,
    }


def build_cost_matrices(var_signatures, site_signatures, metric="cosine", normalize=True):
    var_flat = var_signatures.cpu().numpy()
    site_flat = site_signatures.cpu().numpy()

    C_var = cdist(var_flat, var_flat, metric=metric)
    C_site = cdist(site_flat, site_flat, metric=metric)

    C_var = np.nan_to_num(C_var, nan=0.0, posinf=0.0, neginf=0.0)
    C_site = np.nan_to_num(C_site, nan=0.0, posinf=0.0, neginf=0.0)

    # Enforce symmetry + zero diagonal.
    C_var = 0.5 * (C_var + C_var.T)
    C_site = 0.5 * (C_site + C_site.T)
    np.fill_diagonal(C_var, 0.0)
    np.fill_diagonal(C_site, 0.0)

    # Critical for euclidean stability with entropic GW.
    if normalize:
        var_max = float(C_var.max())
        site_max = float(C_site.max())
        if var_max > 0:
            C_var = C_var / var_max
        if site_max > 0:
            C_site = C_site / site_max

    print(
        f"cost metric={metric}, normalize={normalize}, "
        f"C_var range=[{C_var.min():.3e},{C_var.max():.3e}], "
        f"C_site range=[{C_site.min():.3e},{C_site.max():.3e}]"
    )

    p = np.ones(var_flat.shape[0], dtype=np.float64) / var_flat.shape[0]
    q = np.ones(site_flat.shape[0], dtype=np.float64) / site_flat.shape[0]

    return C_var, C_site, p, q


In [ ]:
effects = build_effect_signatures(alignment_bank, alignment_outputs, TARGET_VARS)
C_var, C_site, p, q = build_cost_matrices(
    effects["var_signatures"],
    effects["site_signatures"],
    metric=GEOMETRY_METRIC,
    normalize=NORMALIZE_COST_MATRICES,
)

print("var_effects shape [V,N,C]:", tuple(effects["var_effects"].shape))
print("site_effects shape [S,N,C]:", tuple(effects["site_effects"].shape))
print("var_effect_mats shape [V,C,N]:", tuple(effects["var_effect_mats"].shape))
print("site_effect_mats shape [S,C,N]:", tuple(effects["site_effect_mats"].shape))
print("var_signatures shape [V,C*N]:", tuple(effects["var_signatures"].shape))
print("site_signatures shape [S,C*N]:", tuple(effects["site_signatures"].shape))
print("C_var shape:", C_var.shape)
print("C_site shape:", C_site.shape)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(C_var, cmap="magma")
axes[0].set_title("Variable geometry C_var")
axes[0].set_xticks(range(len(TARGET_VARS)), TARGET_VARS)
axes[0].set_yticks(range(len(TARGET_VARS)), TARGET_VARS)

axes[1].imshow(C_site, cmap="magma")
axes[1].set_title("Site geometry C_site")
axes[1].set_xlabel("site index")
axes[1].set_ylabel("site index")

cbar0 = fig.colorbar(axes[0].images[0], ax=axes[0], fraction=0.046, pad=0.04)
cbar0.set_label(f"{GEOMETRY_METRIC} distance")

cbar1 = fig.colorbar(axes[1].images[0], ax=axes[1], fraction=0.046, pad=0.04)
cbar1.set_label(f"{GEOMETRY_METRIC} distance")

plt.tight_layout()
plt.show()


## GW Solver (POT)


In [ ]:
def solve_gw(C_var, C_site, p, q, gw_cfg):
    base_eps = gw_cfg.get("epsilon", 1e-2)
    retry_mults = gw_cfg.get("epsilon_retry_multipliers", [1, 5, 10, 50, 100])

    last_T, last_log = None, None
    for mult in retry_mults:
        eps = base_eps * mult
        T, log = ot.gromov.entropic_gromov_wasserstein(
            C_var,
            C_site,
            p,
            q,
            loss_fun=gw_cfg.get("loss_fun", "square_loss"),
            epsilon=eps,
            max_iter=gw_cfg.get("max_iter", 500),
            tol=gw_cfg.get("tol", 1e-9),
            log=True,
            verbose=gw_cfg.get("verbose", False),
        )
        last_T, last_log = T, log

        mass = float(np.sum(T))
        nonzero = int(np.count_nonzero(T))
        finite = bool(np.isfinite(T).all())
        print(f"entropic GW try epsilon={eps:.3e}: mass={mass:.6f}, nonzero={nonzero}, finite={finite}")

        if finite and mass > 0.0 and nonzero > 0:
            return T, {
                "method": "pot_entropic_gw",
                "epsilon_used": eps,
                "log": log,
            }

    print("Warning: all epsilon retries produced degenerate coupling; returning last attempt.")
    return last_T, {
        "method": "pot_entropic_gw_degenerate",
        "epsilon_used": base_eps * retry_mults[-1],
        "log": last_log,
    }


def extract_topk_sites(T, site_specs, var_names, k=3):
    rankings = {}
    for v_idx, var in enumerate(var_names):
        order = np.argsort(-T[v_idx])[:k]
        rankings[var] = [
            {
                "site_index": int(s_idx),
                "site": site_specs[int(s_idx)],
                "score": float(T[v_idx, s_idx]),
            }
            for s_idx in order
        ]
    return rankings


In [ ]:
T, gw_meta = solve_gw(
    C_var=C_var,
    C_site=C_site,
    p=p,
    q=q,
    gw_cfg=GW_CFG,
)

from collections import defaultdict
from matplotlib.gridspec import GridSpec

# Group T columns by layer.
layer_to_cols = defaultdict(list)
for col, site in enumerate(site_specs):
    layer_to_cols[int(site.layer)].append(col)

layers = sorted(layer_to_cols.keys())
max_sites_in_layer = max(len(v) for v in layer_to_cols.values())
vmin, vmax = float(T.min()), float(T.max())

fig = plt.figure(figsize=(max(8, 0.6 * max_sites_in_layer), 2.4 * len(layers)))
gs = GridSpec(
    nrows=len(layers),
    ncols=2,
    width_ratios=[40, 1],   # narrow right column for colorbar
    wspace=0.15,
    hspace=0.45,
)

axes = []
im = None

for r, layer in enumerate(layers):
    ax = fig.add_subplot(gs[r, 0])
    axes.append(ax)

    cols = sorted(layer_to_cols[layer], key=lambda c: site_specs[c].prefix_dim)
    T_layer = T[:, cols]
    x_labels = [f"i={site_specs[c].prefix_dim}" for c in cols]

    im = ax.imshow(T_layer, cmap="viridis", aspect="auto", vmin=vmin, vmax=vmax)
    ax.set_title(f"Layer {layer}")
    ax.set_xticks(range(len(cols)))
    ax.set_xticklabels(x_labels, rotation=90)
    ax.set_yticks(range(len(TARGET_VARS)))
    ax.set_yticklabels(TARGET_VARS)
    ax.set_ylabel("Abstract variable")

axes[-1].set_xlabel("PCA prefix size")

# Single colorbar axis on the very right spanning all rows.
cax = fig.add_subplot(gs[:, 1])
fig.colorbar(im, cax=cax, label="transport mass")

fig.suptitle("Variable-site coupling", y=0.995)
plt.show()


## Held-Out Validation


In [ ]:
def accuracy_for_site(site_cf_logits, target_labels):
    preds = site_cf_logits.argmax(dim=1)
    return float((preds == target_labels).to(torch.float32).mean().item())


def validate_top_matches(T, site_specs, heldout_bank, k=3):
    heldout_outputs = collect_model_outputs(
        trained=trained,
        pair_bank=heldout_bank,
        site_specs=site_specs,
        batch_size=COLLECT_BATCH_SIZE,
        device=device,
    )

    site_cf_logits = heldout_outputs["site_cf_logits"]  # [S, N, C]
    num_sites = site_cf_logits.shape[0]

    rng = np.random.default_rng(SEED + 1234)
    results = {}

    for v_idx, var in enumerate(TARGET_VARS):
        labels = heldout_bank["scm_cf_label_by_var"][var]

        ranked_sites = np.argsort(-T[v_idx])
        topk_sites = ranked_sites[:k]
        top1_site = int(topk_sites[0])

        topk_accs = [accuracy_for_site(site_cf_logits[int(s_idx)], labels) for s_idx in topk_sites]
        top1_acc = topk_accs[0]
        topk_best_acc = max(topk_accs)

        random_site = int(rng.integers(0, num_sites))
        random_acc = accuracy_for_site(site_cf_logits[random_site], labels)

        oracle_accs = [accuracy_for_site(site_cf_logits[s_idx], labels) for s_idx in range(num_sites)]
        oracle_site = int(np.argmax(oracle_accs))
        oracle_best_acc = float(np.max(oracle_accs))

        results[var] = {
            "top1_site": top1_site,
            "top1_acc": top1_acc,
            "topk_sites": [int(x) for x in topk_sites],
            "topk_best_acc": topk_best_acc,
            "random_site": random_site,
            "random_acc": random_acc,
            "oracle_site": oracle_site,
            "oracle_best_acc": oracle_best_acc,
        }

    return results


In [ ]:
validation_results = validate_top_matches(T, site_specs, heldout_bank, k=TOP_K_VALIDATE)

print("Held-out validation summary")
for var in TARGET_VARS:
    row = validation_results[var]
    top_site = site_specs[row["top1_site"]]
    print(
        f"{var}: top1=L{top_site.layer}-i{top_site.prefix_dim} acc={row['top1_acc']:.4f} | "
        f"top{TOP_K_VALIDATE}_best={row['topk_best_acc']:.4f} | "
        f"random={row['random_acc']:.4f} | oracle={row['oracle_best_acc']:.4f}"
    )


## Test and Scenario Checks


In [ ]:
# 1) SCM sanity already checked above.
spot = addition_model.run_forward({
    "A1": one_hot_digits[9],
    "B1": one_hot_digits[8],
    "A2": one_hot_digits[9],
    "B2": one_hot_digits[9],
})
assert int(spot["S1"]) == 7 and int(spot["C1"]) == 1
assert int(spot["S2"]) == 9 and int(spot["C2"]) == 1
assert int(spot["O"]) == 197

# 2) Data integrity checks.
assert alignment_bank["base_inputs"].shape[0] == N_ALIGNMENT_PAIRS
assert heldout_bank["base_inputs"].shape[0] == N_HELDOUT_PAIRS

# 3) Intervention pipeline checks.
assert alignment_outputs["site_cf_logits"].shape[0] == len(site_specs)
assert not torch.isnan(alignment_outputs["site_cf_logits"]).any()

# 4) GW checks.
assert T.shape == (len(TARGET_VARS), len(site_specs))
assert np.allclose(T.sum(axis=1), p, atol=1e-4)
assert np.allclose(T.sum(axis=0), q, atol=1e-4)

# 5) Validation checks.
assert set(validation_results.keys()) == set(TARGET_VARS)
for var in TARGET_VARS:
    row = validation_results[var]
    assert len(row["topk_sites"]) == TOP_K_VALIDATE

print("All checks passed.")
